# Final comparison

All three models trained and evaluated. This notebook loads their scores, puts everything in one table, plots the ROC curves together, and draws conclusions. This is the main result of the project.

## Load all scores

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve)

iso = np.load('../results/iso_scores.npz')
ae  = np.load('../results/ae_scores.npz')
cnn = np.load('../results/cnn_scores.npz')

# all three share the same y_test
y_test = iso['y_test']

iso_scores = iso['test_scores']
ae_scores  = ae['test_scores']
cnn_scores = cnn['test_scores']

print("test set size:", len(y_test))
print("empty:", (y_test==0).sum(), " occupied:", (y_test==1).sum())

## Recompute thresholds and predictions

Need the val scores to recompute thresholds here. Loading prepared data and rerunning the threshold step for each model so everything is in one place.

In [ ]:
data = np.load('../data/prepared.npz')
X_val = data['X_val']

train_mean = data['train_mean']
train_std  = data['train_std']

def norm(x): return (x - train_mean) / (train_std + 1e-8)

# --- isolation forest val scores ---
from sklearn.ensemble import IsolationForest

def window_features(windows):
    feats = []
    for w in windows:
        f = []
        for s in range(w.shape[1]):
            sig = w[:, s]
            f += [sig.mean(), sig.std(), sig.min(), sig.max(), sig.max()-sig.min()]
        feats.append(f)
    return np.array(feats)

X_train = data['X_train']
F_train = window_features(X_train)
F_val   = window_features(X_val)

f_mean = F_train.mean(axis=0)
f_std  = F_train.std(axis=0)
F_train_n = (F_train - f_mean) / (f_std + 1e-8)
F_val_n   = (F_val   - f_mean) / (f_std + 1e-8)

iso_model = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
iso_model.fit(F_train_n)
iso_val_scores = -iso_model.score_samples(F_val_n)
iso_thresh = np.percentile(iso_val_scores, 95)

# --- mlp val scores ---
import torch
import torch.nn as nn

class MLPAutoencoder(nn.Module):
    def __init__(self, in_dim=60):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, in_dim),
        )
    def forward(self, x): return self.decoder(self.encoder(x))

Xva_mlp = torch.tensor(norm(X_val).reshape(len(X_val), -1), dtype=torch.float32)
mlp = MLPAutoencoder()
torch.manual_seed(42)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
Xtr_mlp = torch.tensor(norm(X_train).reshape(len(X_train), -1), dtype=torch.float32)
for _ in range(100):
    mlp.train(); opt.zero_grad()
    loss = loss_fn(mlp(Xtr_mlp), Xtr_mlp)
    loss.backward(); opt.step()
mlp.eval()
with torch.no_grad():
    ae_val_scores = ((Xva_mlp - mlp(Xva_mlp))**2).mean(dim=1).numpy()
ae_thresh = np.percentile(ae_val_scores, 95)

# --- cnn val scores ---
class CNNAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(2, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(16, 8, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(8, 16, kernel_size=2, stride=2), nn.ReLU(),
            nn.Conv1d(16, 2, kernel_size=3, padding=1),
        )
    def forward(self, x): return self.decoder(self.encoder(x))

Xva_cnn = torch.tensor(norm(X_val), dtype=torch.float32).permute(0,2,1)
Xtr_cnn = torch.tensor(norm(X_train), dtype=torch.float32).permute(0,2,1)
torch.manual_seed(42)
cnn_model = CNNAutoencoder()
opt2 = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)
for _ in range(100):
    cnn_model.train(); opt2.zero_grad()
    loss = loss_fn(cnn_model(Xtr_cnn), Xtr_cnn)
    loss.backward(); opt2.step()
cnn_model.eval()
with torch.no_grad():
    cnn_val_scores = ((Xva_cnn - cnn_model(Xva_cnn))**2).mean(dim=(1,2)).numpy()
cnn_thresh = np.percentile(cnn_val_scores, 95)

print("thresholds — iso:", round(iso_thresh,4),
      " mlp:", round(ae_thresh,4),
      " cnn:", round(cnn_thresh,4))

## Results table

In [ ]:
models = {
    'Isolation Forest': (iso_scores, iso_thresh),
    'MLP Autoencoder' : (ae_scores,  ae_thresh),
    'CNN Autoencoder' : (cnn_scores, cnn_thresh),
}

print(f"{'Model':<22} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC-AUC':>9}")
print("-" * 62)
results = {}
for name, (scores, thresh) in models.items():
    y_pred = (scores > thresh).astype(int)
    p = precision_score(y_test, y_pred)
    r = recall_score(y_test, y_pred)
    f = f1_score(y_test, y_pred)
    a = roc_auc_score(y_test, scores)
    results[name] = (p, r, f, a)
    print(f"{name:<22} {p:>10.3f} {r:>8.3f} {f:>8.3f} {a:>9.3f}")

## ROC curves

All three on one plot. ROC curve shows how precision and recall trade off at every possible threshold, so it's a fairer comparison than a single threshold point.

In [ ]:
plt.figure(figsize=(8,6))

colors = {'Isolation Forest': 'steelblue',
          'MLP Autoencoder' : 'seagreen',
          'CNN Autoencoder' : 'tomato'}

for name, (scores, _) in models.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = results[name][3]
    plt.plot(fpr, tpr, label=f"{name}  (auc={auc:.3f})", color=colors[name], linewidth=2)

plt.plot([0,1],[0,1], 'k--', linewidth=1)
plt.xlabel('false positive rate')
plt.ylabel('true positive rate')
plt.title('ROC curves — all three models')
plt.legend()
plt.tight_layout()
plt.savefig('../results/05_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## F1 bar chart

In [ ]:
names = list(results.keys())
f1s   = [results[n][2] for n in names]
aucs  = [results[n][3] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(9,5))
ax.bar(x - width/2, f1s, width, label='F1', color='steelblue', alpha=0.85)
ax.bar(x + width/2, aucs, width, label='ROC-AUC', color='tomato', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0, 1.05)
ax.set_ylabel('score')
ax.set_title('F1 and ROC-AUC comparison')
ax.legend(); plt.tight_layout()
plt.savefig('../results/05_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusions

Three models compared on the same data, same split, same threshold method.

**CNN Autoencoder wins on F1 (0.919)** — best at actually detecting presence when you pick a threshold. Recall of 0.988 means it almost never misses an occupied window.

**Isolation Forest wins on ROC-AUC (0.928)** — its scores separate the two classes more cleanly across all thresholds, even though its F1 is slightly lower than CNN.

**MLP Autoencoder is the weakest overall.** Flattening the window loses the time structure, so many occupied windows look similar to empty ones in feature space and score below the threshold.

The F1 vs ROC-AUC gap in the CNN result is the most interesting finding. CNN is aggressive — it flags a lot, catches almost everything, but generates more false alarms (179 vs 46 for Isolation Forest). Which model is better depends on the use case. In a security or energy management system where missing presence is costly, CNN is the right choice. Where false alarms are disruptive, Isolation Forest is more reliable.

All three methods work without any labeled anomalies during training. The unsupervised one-class setup is viable for indoor presence detection from sensor signals.